# Streaming Syslog Fault Detection for SmartNIC-ICS

This notebook builds a fault detector for industrial-control syslogs using two paths:

1. **Python research path**: parse logs, build 1-second windows, train/evaluate a model, replay logs as if they are being appended live.
2. **SmartNIC CPU-portable path**: train a lightweight model based on fixed numeric features and export its weights/feature order so the same logic can be rewritten in C for SmartNIC CPU cores.

The notebook uses your log locations:

```python
fault_df = parse_file("/home/ubuntu/raw_data/fault/all_logs.log", label_name="fault", label_value=1)
normal_df = parse_file("/home/ubuntu/raw_data/normal/all_logs.log", label_name="normal", label_value=0)
```

Labels:

- `0` = normal
- `1` = fault

Important design choice: for actual SmartNIC deployment, the most portable model is **not a transformer**. A transformer/small language model is useful for offline research, but for C on SmartNIC CPU cores, a fixed-feature model such as logistic regression is much easier to port, audit, and run in real time.

## 1. Dependency setup

The notebook has two paths:

- **SmartNIC-portable path, recommended:** uses `pandas`, `numpy`, `scikit-learn`, and `joblib` only. This is the path that exports fixed coefficients you can port to C.
- **Optional transformer research path:** uses Hugging Face packages. Do **not** run this inside an older system Python environment unless you use a clean virtual environment or Conda environment.

Your error about `packaging` and `requests` means the current Python environment has old system packages. The safest fix is to avoid installing `transformers/datasets` in-place. The SmartNIC-portable model below does not need them.


In [ ]:
# Core dependencies for the SmartNIC-portable path.
# Run this only if imports fail in the next cell.
# It intentionally does NOT install transformers/datasets.

# !python -m pip install --upgrade pip
# !python -m pip install --upgrade pandas numpy scikit-learn joblib

# Optional research-only transformer environment.
# Recommended: create a clean virtual environment instead of modifying system Python.
# Example from a terminal, not from the SmartNIC target:
#   python3 -m venv ~/venvs/syslog-slm
#   source ~/venvs/syslog-slm/bin/activate
#   python -m pip install --upgrade pip setuptools wheel
#   python -m pip install "packaging>=20.9" "requests>=2.32.2"
#   python -m pip install torch transformers accelerate datasets
#   python -m ipykernel install --user --name syslog-slm --display-name "Python (syslog-slm)"

print("For the SmartNIC-portable path, continue without installing Hugging Face packages.")


In [1]:
# Environment diagnostic: useful when dependency conflicts appear.
import sys
print("Python:", sys.version)

try:
    import packaging
    print("packaging:", packaging.__version__)
except Exception as e:
    print("packaging not importable:", e)

try:
    import requests
    print("requests:", requests.__version__)
except Exception as e:
    print("requests not importable:", e)


Python: 3.8.10 (default, Mar 18 2025, 20:04:55) 
[GCC 9.4.0]
packaging: 20.3
requests: 2.22.0


## 2. Imports and configuration

In [2]:
import os
import re
import time
import json
import math
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

In [3]:
FAULT_LOG_PATH = "/home/ubuntu/raw_data/fault/all_logs.log"
NORMAL_LOG_PATH = "/home/ubuntu/raw_data/normal/all_logs.log"

OUTPUT_DIR = Path("./syslog_fault_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
WINDOW_SIZE = "1s"   # one-second detection windows

# Fault keyword list is used only for interpretable features, not for weak labeling here.
# Labels come from the file source: normal file = 0, fault file = 1.
FAULT_KEYWORDS = [
    "fault",
    "warning",
    "error",
    "critical",
    "sensor drift",
    "tank leak",
    "memory corruption",
    "conveyor belt sticking",
    "none",
    "null",
    "nonetype",
    "attacker",
    "ddos",
    "mitm",
    "replay",
    "command-injection",
    "scan-nmap",
    "scan-scapy",
]

print("Configured paths:")
print("Fault log: ", FAULT_LOG_PATH)
print("Normal log:", NORMAL_LOG_PATH)

Configured paths:
Fault log:  /home/ubuntu/raw_data/fault/all_logs.log
Normal log: /home/ubuntu/raw_data/normal/all_logs.log


## 3. Syslog parsing

This parser keeps the raw line, the first syslog timestamp, the message body, and basic metadata.

It is intentionally simple because the same parsing logic can later be implemented in C:

- read line
- extract first timestamp if present
- scan for severity strings
- scan for device/component strings
- count keywords

In [4]:
# Example syslog prefix:
# 2025-01-10T03:22:34.616806+08:00 rest of message
TIMESTAMP_PATTERN = re.compile(
    r"^(?P<ts>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}:\d{2}))\s+(?P<msg>.*)$"
)

# Some inner app logs contain patterns like: PLC1 WARNING: message
SEVERITY_PATTERN = re.compile(r"\b(DEBUG|INFO|WARNING|ERROR|CRITICAL)\b", re.IGNORECASE)
COMPONENT_PATTERN = re.compile(r"\b(PLC\d+|HMI\d+|attacker(?:remote|machine)?|rsyslogd|snapshots_PLC\d+)\b", re.IGNORECASE)
IP_PATTERN = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
NUMBER_PATTERN = re.compile(r"[-+]?\d*\.\d+|[-+]?\d+")
ANSI_PATTERN = re.compile(r"#033\[[0-9;]*[A-Za-z]")


def clean_message(text: str) -> str:
    """Remove common escaped terminal noise while preserving useful words."""
    if not isinstance(text, str):
        return ""
    text = ANSI_PATTERN.sub(" ", text)
    text = text.replace("#012", " ").replace("#011", " ").replace("#015", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_line(line: str):
    line = line.rstrip("\n")
    match = TIMESTAMP_PATTERN.match(line)

    if match:
        ts_raw = match.group("ts")
        msg = match.group("msg")
        try:
            ts = pd.to_datetime(ts_raw, utc=True, errors="coerce")
        except Exception:
            ts = pd.NaT
    else:
        ts_raw = None
        ts = pd.NaT
        msg = line

    msg_clean = clean_message(msg)

    sev_match = SEVERITY_PATTERN.search(msg_clean)
    comp_match = COMPONENT_PATTERN.search(msg_clean)

    return {
        "timestamp": ts,
        "timestamp_raw": ts_raw,
        "raw_line": line,
        "message": msg_clean,
        "severity": sev_match.group(1).upper() if sev_match else "UNKNOWN",
        "component": comp_match.group(1).upper() if comp_match else "UNKNOWN",
        "has_ip": int(bool(IP_PATTERN.search(msg_clean))),
        "num_numeric_tokens": len(NUMBER_PATTERN.findall(msg_clean)),
    }


def parse_file(path: str, label_name: str, label_value: int) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find log file: {path}")

    rows = []
    with path.open("r", errors="ignore") as f:
        for line_id, line in enumerate(f):
            if not line.strip():
                continue
            row = parse_line(line)
            row["line_id"] = line_id
            row["source_file"] = str(path)
            row["label_name"] = label_name
            row["label"] = int(label_value)
            rows.append(row)

    df = pd.DataFrame(rows)
    return df

## 4. Load your normal and fault logs

In [5]:
fault_df = parse_file("/home/ubuntu/raw_data/fault/all_logs.log", label_name="fault", label_value=1)
normal_df = parse_file("/home/ubuntu/raw_data/normal/all_logs.log", label_name="normal", label_value=0)

line_df = pd.concat([normal_df, fault_df], ignore_index=True)

print("Normal lines:", len(normal_df))
print("Fault lines: ", len(fault_df))
print("Total lines: ", len(line_df))
line_df.head()

Normal lines: 1102464
Fault lines:  494662
Total lines:  1597126


,timestamp,timestamp_raw,raw_line,message,severity,component,has_ip,num_numeric_tokens,line_id,source_file,label_name,label
0,2025-01-09 14:43:07.159018+00:00,2025-01-09T22:43:07.159018+08:00,2025-01-09T22:43:07.159018+08:00 15b16d2baaa8 ...,15b16d2baaa8 rsyslogd: environment variable TZ...,UNKNOWN,RSYSLOGD,0,7,0,/home/ubuntu/raw_data/normal/all_logs.log,normal,0
1,2025-01-09 14:43:07.159020+00:00,2025-01-09T22:43:07.159020+08:00,2025-01-09T22:43:07.159020+08:00 15b16d2baaa8 ...,15b16d2baaa8 rsyslogd: rsyslogd's groupid chan...,UNKNOWN,RSYSLOGD,0,5,1,/home/ubuntu/raw_data/normal/all_logs.log,normal,0
2,2025-01-09 14:43:07.159021+00:00,2025-01-09T22:43:07.159021+08:00,2025-01-09T22:43:07.159021+08:00 15b16d2baaa8 ...,15b16d2baaa8 rsyslogd: rsyslogd's userid chang...,UNKNOWN,RSYSLOGD,0,5,2,/home/ubuntu/raw_data/normal/all_logs.log,normal,0
3,2025-01-09 14:43:07.159022+00:00,2025-01-09T22:43:07.159022+08:00,2025-01-09T22:43:07.159022+08:00 15b16d2baaa8 ...,"15b16d2baaa8 rsyslogd: [origin software=""rsysl...",INFO,RSYSLOGD,0,7,3,/home/ubuntu/raw_data/normal/all_logs.log,normal,0
4,2025-01-09 14:43:07.304726+00:00,2025-01-09T22:43:07.304726+08:00,2025-01-09T22:43:07.304726+08:00 2025-01-09 22...,"2025-01-09 22: 43:06,277 logs-HMI3 INFO: [HMI3...",INFO,HMI3,0,9,4,/home/ubuntu/raw_data/normal/all_logs.log,normal,0


## 5. Build one-second windows

The detection unit is a **one-second window**.

Each row below represents the log lines that arrived during one second. The label is inherited from the source file: windows from the normal log are normal, and windows from the fault log are fault.

In [6]:
def make_windows(df: pd.DataFrame, window_size: str = "1s") -> pd.DataFrame:
    df = df.copy()

    # For lines with missing timestamps, create a synthetic timeline using line order.
    # This avoids losing data, but real deployment should always use the arrival time.
    if df["timestamp"].isna().all():
        base = pd.Timestamp("1970-01-01", tz="UTC")
        df["timestamp"] = [base + pd.Timedelta(seconds=i) for i in range(len(df))]
    else:
        # Fill rare missing timestamps with previous timestamp, then next timestamp.
        df["timestamp"] = df["timestamp"].ffill().bfill()

    df = df.sort_values(["source_file", "timestamp", "line_id"])
    df["window_start"] = df["timestamp"].dt.floor(window_size)

    grouped = (
        df.groupby(["source_file", "label", "label_name", "window_start"], dropna=False)
        .agg(
            window_text=("message", lambda x: "\n".join(x.astype(str))),
            raw_window_text=("raw_line", lambda x: "\n".join(x.astype(str))),
            num_lines=("message", "count"),
            num_ips=("has_ip", "sum"),
            num_numeric_tokens=("num_numeric_tokens", "sum"),
        )
        .reset_index()
    )

    return grouped

window_df = make_windows(line_df, WINDOW_SIZE)

print("Windows:", len(window_df))
print(window_df["label_name"].value_counts())
window_df.head()

Windows: 25900
label_name
normal    16400
fault      9500
Name: count, dtype: int64


,source_file,label,label_name,window_start,window_text,raw_window_text,num_lines,num_ips,num_numeric_tokens
0,/home/ubuntu/raw_data/fault/all_logs.log,1,fault,2025-01-09 19:22:32+00:00,smasher-virtualbox attackerremote[826]: [ atta...,2025-01-10T03:22:32+08:00 smasher-virtualbox a...,20,0,31
1,/home/ubuntu/raw_data/fault/all_logs.log,1,fault,2025-01-09 19:22:33+00:00,smasher-virtualbox hmi3[826]: [ HMI3 - 03:22:3...,2025-01-10T03:22:33+08:00 smasher-virtualbox h...,26,0,88
2,/home/ubuntu/raw_data/fault/all_logs.log,1,fault,2025-01-09 19:22:34+00:00,smasher-virtualbox hmi1[826]: [ HMI1 - 03:22:3...,2025-01-10T03:22:34+08:00 smasher-virtualbox h...,110,3,733
3,/home/ubuntu/raw_data/fault/all_logs.log,1,fault,2025-01-09 19:22:35+00:00,smasher-virtualbox hmi1[826]: [ HMI1 - 03:22:3...,2025-01-10T03:22:35+08:00 smasher-virtualbox h...,39,0,132
4,/home/ubuntu/raw_data/fault/all_logs.log,1,fault,2025-01-09 19:22:36+00:00,smasher-virtualbox hmi1[826]: [ HMI1 - 03:22:3...,2025-01-10T03:22:36+08:00 smasher-virtualbox h...,39,0,127


## 6. Feature extraction for a C-portable model

These features are deliberately simple:

- line count in the last second
- severity counts
- component counts
- keyword counts
- number/IP counts
- basic message length statistics

This is the feature set you can port to C on SmartNIC CPU cores.

In [7]:
def count_occurrences(text: str, phrase: str) -> int:
    return str(text).lower().count(phrase.lower())


def extract_c_portable_features(window_text: str, num_lines=None, num_ips=None, num_numeric_tokens=None) -> dict:
    text = str(window_text)
    lower = text.lower()
    lines = [ln for ln in text.splitlines() if ln.strip()]

    if num_lines is None:
        num_lines = len(lines)
    if num_ips is None:
        num_ips = len(IP_PATTERN.findall(text))
    if num_numeric_tokens is None:
        num_numeric_tokens = len(NUMBER_PATTERN.findall(text))

    features = {}
    features["num_lines"] = float(num_lines)
    features["num_chars"] = float(len(text))
    features["avg_line_len"] = float(len(text) / max(num_lines, 1))
    features["num_ips"] = float(num_ips)
    features["num_numeric_tokens"] = float(num_numeric_tokens)

    # Severity counts
    for sev in ["debug", "info", "warning", "error", "critical"]:
        features[f"count_{sev}"] = float(count_occurrences(lower, sev))

    # Component/device counts
    for comp in ["plc1", "plc2", "hmi1", "hmi2", "hmi3", "attacker", "rsyslogd", "snapshot"]:
        features[f"count_{comp}"] = float(count_occurrences(lower, comp))

    # Domain/fault keyword counts
    for kw in FAULT_KEYWORDS:
        safe_kw = re.sub(r"[^a-zA-Z0-9]+", "_", kw.lower()).strip("_")
        features[f"kw_{safe_kw}"] = float(count_occurrences(lower, kw))

    return features


def build_feature_table(windows: pd.DataFrame):
    feature_rows = []
    for _, row in windows.iterrows():
        feature_rows.append(
            extract_c_portable_features(
                row["window_text"],
                num_lines=row.get("num_lines"),
                num_ips=row.get("num_ips"),
                num_numeric_tokens=row.get("num_numeric_tokens"),
            )
        )
    X = pd.DataFrame(feature_rows).fillna(0.0)
    return X

X_features = build_feature_table(window_df)
y = window_df["label"].astype(int).values

print("Feature matrix:", X_features.shape)
X_features.head()

Feature matrix: (25900, 36)


,num_lines,num_chars,avg_line_len,num_ips,num_numeric_tokens,count_debug,count_info,count_warning,count_error,count_critical,...,kw_none,kw_null,kw_nonetype,kw_attacker,kw_ddos,kw_mitm,kw_replay,kw_command_injection,kw_scan_nmap,kw_scan_scapy
0,20.0,1477.0,73.850000,0.0,31.0,0.0,2.0,1.0,0.0,0.0,...,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0
1,26.0,1863.0,71.653846,0.0,88.0,0.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,19.0,1.0,1.0,1.0,1.0,1.0,1.0
2,110.0,9697.0,88.154545,3.0,733.0,0.0,35.0,31.0,14.0,1.0,...,28.0,13.0,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,39.0,2991.0,76.692308,0.0,132.0,0.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,39.0,3007.0,77.102564,0.0,127.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Train/test split

In [8]:
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X_features,
    y,
    np.arange(len(window_df)),
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y if len(np.unique(y)) > 1 else None,
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)
print("Train labels:", Counter(y_train))
print("Test labels: ", Counter(y_test))

Train size: (19425, 36)
Test size:  (6475, 36)
Train labels: Counter({0: 12300, 1: 7125})
Test labels:  Counter({0: 4100, 1: 2375})


## 8. Train the SmartNIC-portable model

This model is intentionally simple:

```text
score = bias + sum(weight_i * normalized_feature_i)
probability = 1 / (1 + exp(-score))
```

That exact computation is easy to transfer to C.

In [9]:
portable_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

portable_model.fit(X_train, y_train)

y_pred = portable_model.predict(X_test)
y_prob = portable_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["normal", "fault"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

if len(np.unique(y_test)) > 1:
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      4100
       fault       1.00      1.00      1.00      2375

    accuracy                           1.00      6475
   macro avg       1.00      1.00      1.00      6475
weighted avg       1.00      1.00      1.00      6475

Confusion matrix:
[[4100    0]
 [   1 2374]]
ROC AUC: 0.999578947368421


## 9. Inspect important features

In [10]:
scaler = portable_model.named_steps["scaler"]
clf = portable_model.named_steps["clf"]

coef = clf.coef_[0]
feature_names = list(X_features.columns)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coef,
    "abs_coefficient": np.abs(coef),
}).sort_values("abs_coefficient", ascending=False)

importance_df.head(25)

,feature,coefficient,abs_coefficient
11,count_plc2,-1.761263,1.761263
6,count_info,-1.400979,1.400979
4,num_numeric_tokens,-1.195264,1.195264
10,count_plc1,-1.072180,1.072180
17,count_snapshot,-1.037033,1.037033
18,kw_fault,0.882257,0.882257
2,avg_line_len,-0.761377,0.761377
8,count_error,0.652903,0.652903
20,kw_error,0.652903,0.652903
9,count_critical,0.651630,0.651630


## 10. Save Python model and export C-portable parameters

The `.joblib` file is useful for Python.

The `.json` export is the important file for C. It contains:

- feature names in exact order
- mean and standard deviation from `StandardScaler`
- logistic regression weights
- logistic regression bias
- suggested threshold

In [11]:
MODEL_JOBLIB_PATH = OUTPUT_DIR / "portable_syslog_fault_model.joblib"
EXPORT_JSON_PATH = OUTPUT_DIR / "portable_syslog_fault_model_export.json"
FEATURES_TXT_PATH = OUTPUT_DIR / "feature_order.txt"

joblib.dump(portable_model, MODEL_JOBLIB_PATH)

export = {
    "model_type": "standardized_logistic_regression",
    "label_mapping": {"0": "normal", "1": "fault"},
    "window_size": WINDOW_SIZE,
    "threshold": 0.5,
    "feature_order": feature_names,
    "scaler_mean": scaler.mean_.astype(float).tolist(),
    "scaler_scale": scaler.scale_.astype(float).tolist(),
    "weights": clf.coef_[0].astype(float).tolist(),
    "bias": float(clf.intercept_[0]),
    "fault_keywords": FAULT_KEYWORDS,
}

with EXPORT_JSON_PATH.open("w") as f:
    json.dump(export, f, indent=2)

with FEATURES_TXT_PATH.open("w") as f:
    for name in feature_names:
        f.write(name + "\n")

print("Saved:")
print(MODEL_JOBLIB_PATH)
print(EXPORT_JSON_PATH)
print(FEATURES_TXT_PATH)

Saved:
syslog_fault_outputs/portable_syslog_fault_model.joblib
syslog_fault_outputs/portable_syslog_fault_model_export.json
syslog_fault_outputs/feature_order.txt


## 11. Pure-Python inference equivalent to C

This function does not use scikit-learn. It performs the exact logistic regression math directly.

This is the logic you would port to C.

In [12]:
def sigmoid_stable(x: float) -> float:
    # Numerically stable sigmoid. Easy to implement in C using exp().
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    else:
        z = math.exp(x)
        return z / (1.0 + z)


def predict_fault_probability_exported(feature_dict: dict, export_dict: dict) -> float:
    score = float(export_dict["bias"])

    for i, name in enumerate(export_dict["feature_order"]):
        x = float(feature_dict.get(name, 0.0))
        mean = float(export_dict["scaler_mean"][i])
        scale = float(export_dict["scaler_scale"][i])
        weight = float(export_dict["weights"][i])

        if scale == 0.0:
            x_scaled = 0.0
        else:
            x_scaled = (x - mean) / scale

        score += weight * x_scaled

    return sigmoid_stable(score)


def classify_window_c_equivalent(window_text: str, export_dict: dict, threshold: float = None):
    if threshold is None:
        threshold = float(export_dict.get("threshold", 0.5))

    feats = extract_c_portable_features(window_text)
    prob = predict_fault_probability_exported(feats, export_dict)
    pred = int(prob >= threshold)

    return {
        "prediction": "fault" if pred == 1 else "normal",
        "fault_probability": prob,
        "features": feats,
    }

# Quick consistency check against scikit-learn for a few test rows
with EXPORT_JSON_PATH.open("r") as f:
    loaded_export = json.load(f)

for local_i in range(min(5, len(X_test))):
    row_features = X_test.iloc[local_i].to_dict()
    p_export = predict_fault_probability_exported(row_features, loaded_export)
    p_sklearn = portable_model.predict_proba(X_test.iloc[[local_i]])[0, 1]
    print(local_i, "exported=", round(p_export, 6), "sklearn=", round(float(p_sklearn), 6))

0 exported= 6e-06 sklearn= 6e-06
1 exported= 0.999997 sklearn= 0.999997
2 exported= 6e-06 sklearn= 6e-06
3 exported= 6e-06 sklearn= 6e-06
4 exported= 0.999998 sklearn= 0.999998


## 12. Replay historical logs as if they are being appended live

This simulates a live log stream. Each iteration returns the lines belonging to the next second.

In [13]:
def replay_log_by_second(df: pd.DataFrame, sleep_time: float = 1.0):
    df_sorted = df.copy()
    df_sorted["timestamp"] = df_sorted["timestamp"].ffill().bfill()
    df_sorted = df_sorted.sort_values(["timestamp", "line_id"])
    df_sorted["window_start"] = df_sorted["timestamp"].dt.floor("1s")

    for second, group in df_sorted.groupby("window_start"):
        lines = group["raw_line"].tolist()
        yield second, lines
        if sleep_time and sleep_time > 0:
            time.sleep(sleep_time)


def classify_lines(lines, export_dict, threshold=0.5):
    text = "\n".join(lines)
    return classify_window_c_equivalent(text, export_dict, threshold=threshold)

In [17]:
FAULT_EVENT_KEYWORDS = [
    "fault applied",
    "sensor drift applied",
    "tank leak fault",
    "conveyor belt sticking fault",
    "memory corruption fault",
    "actuator failure",
    "fault",
    "critical",
    "error",
]

def infer_window_ground_truth(lines):
    text = "\n".join(lines).lower()
    return 1 if any(k in text for k in FAULT_EVENT_KEYWORDS) else 0

for second, new_lines in replay_log_by_second(fault_df, sleep_time=0.0):
    result = classify_lines(new_lines, loaded_export, threshold=0.5)

    true_label = infer_window_ground_truth(new_lines)
    true_name = "fault" if true_label == 1 else "normal/no-fault-yet"

    pred_label = 1 if result["prediction"] == "fault" else 0

    print("=" * 100)
    print("Window:", second)
    print("Source file:", "fault_df")
    print("Window ground truth:", true_name)
    print("Lines:", len(new_lines))
    print("Prediction:", result["prediction"])
    print("Correct:", pred_label == true_label)
    print("Fault probability:", round(result["fault_probability"], 4))
    print("Sample lines:")
    print("\n".join(new_lines[:3]))
    break

Window: 2025-01-09 19:22:32+00:00
Source file: fault_df
Window ground truth: normal/no-fault-yet
Lines: 20
Prediction: normal
Correct: True
Fault probability: 0.0
Sample lines:
2025-01-10T03:22:32+08:00 smasher-virtualbox attackerremote[826]: [#033attacker_remote#033 - #03303:22:32#033[0m]#011#033[INFO] Created#033#015
2025-01-10T03:22:32+08:00 smasher-virtualbox attackerremote[826]: AttackerRemote.py:25: DeprecationWarning: Callback API version 1 is deprecated, update to latest version#015
2025-01-10T03:22:32+08:00 smasher-virtualbox attackerremote[826]:   self.client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION1)#015


In [22]:
FAULT_EVENT_KEYWORDS = [
    "fault applied",
    "sensor drift applied",
    "tank leak fault",
    "conveyor belt sticking fault",
    "memory corruption fault",
    "actuator failure",
    "fault",
    "critical",
    "error",
]

def infer_window_ground_truth(lines):
    text = "\n".join(lines).lower()
    return 1 if any(k in text for k in FAULT_EVENT_KEYWORDS) else 0


for second, new_lines in replay_log_by_second(normal_df, sleep_time=0.0):
    result = classify_lines(new_lines, loaded_export, threshold=0.5)

    true_label = infer_window_ground_truth(new_lines)
    true_name = "fault" if true_label == 1 else "normal"

    pred_label = 1 if result["prediction"] == "fault" else 0

    print("=" * 100)
    print("Window:", second)
    print("Source file:", "normal_df")
    print("Window ground truth:", true_name)
    print("Lines:", len(new_lines))
    print("Prediction:", result["prediction"])
    print("Correct:", pred_label == true_label)
    print("Fault probability:", round(result["fault_probability"], 4))
    print("Sample lines:")
    print("\n".join(new_lines[:3]))
    break

Window: 2025-01-09 14:43:05+00:00
Source file: normal_df
Window ground truth: normal
Lines: 1
Prediction: normal
Correct: True
Fault probability: 0.0011
Sample lines:
2025-01-09T22:43:05+08:00 smasher-virtualbox pys[826]: Starting memcached: memcached.


## Testing

In [18]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

FAULT_EVENT_KEYWORDS = [
    "fault applied",
    "sensor drift applied",
    "tank leak fault",
    "conveyor belt sticking fault",
    "memory corruption fault",
    "actuator failure",
    "fault",
    "critical",
    "error",
]

def infer_window_ground_truth(lines):
    text = "\n".join(lines).lower()
    return 1 if any(k in text for k in FAULT_EVENT_KEYWORDS) else 0


def evaluate_replay_windows(
    source_df,
    source_file_name,
    loaded_export,
    threshold=0.5,
    sleep_time=0.0,
    print_each_window=False,
    max_sample_lines=3,
):
    """
    Replay all windows from a parsed log dataframe and compare:
      - true_label: inferred from FAULT_EVENT_KEYWORDS
      - pred_label: predicted by classify_lines()

    Returns a dataframe with one row per 1-second window.
    """

    rows = []

    for second, new_lines in replay_log_by_second(source_df, sleep_time=sleep_time):
        if not new_lines:
            continue

        result = classify_lines(new_lines, loaded_export, threshold=threshold)

        true_label = infer_window_ground_truth(new_lines)
        pred_label = 1 if result["prediction"] == "fault" else 0

        if true_label == 1 and pred_label == 1:
            outcome = "true_positive"
        elif true_label == 0 and pred_label == 0:
            outcome = "true_negative"
        elif true_label == 0 and pred_label == 1:
            outcome = "false_positive"
        else:
            outcome = "false_negative"

        row = {
            "second": second,
            "source_file": source_file_name,
            "num_lines": len(new_lines),
            "true_label": true_label,
            "true_name": "fault" if true_label == 1 else "normal/no-fault",
            "pred_label": pred_label,
            "prediction": result["prediction"],
            "fault_probability": result["fault_probability"],
            "correct": pred_label == true_label,
            "outcome": outcome,
            "sample_lines": "\n".join(new_lines[:max_sample_lines]),
        }

        rows.append(row)

        if print_each_window:
            print("=" * 100)
            print("Window:", second)
            print("Source file:", source_file_name)
            print("Window ground truth:", row["true_name"])
            print("Lines:", len(new_lines))
            print("Prediction:", result["prediction"])
            print("Correct:", row["correct"])
            print("Fault probability:", round(result["fault_probability"], 4))
            print("Outcome:", outcome)
            print("Sample lines:")
            print(row["sample_lines"])

    results_df = pd.DataFrame(rows)

    if results_df.empty:
        print("No replay windows were evaluated.")
        return results_df

    y_true = results_df["true_label"]
    y_pred = results_df["pred_label"]

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    total = len(results_df)
    correct = int((results_df["correct"] == True).sum())
    incorrect = total - correct

    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0

    print("=" * 100)
    print("FINAL REPLAY SUMMARY")
    print("=" * 100)
    print("Source file:", source_file_name)
    print("Threshold:", threshold)
    print("Total 1-second windows:", total)
    print("Correct windows:", correct)
    print("Incorrect windows:", incorrect)
    print()
    print("Confusion matrix counts:")
    print("True negatives :", tn)
    print("False positives:", fp)
    print("False negatives:", fn)
    print("True positives :", tp)
    print()
    print("Rates:")
    print("Accuracy           :", round(accuracy, 4))
    print("Precision          :", round(precision, 4))
    print("Recall             :", round(recall, 4))
    print("False positive rate:", round(false_positive_rate, 4))
    print("False negative rate:", round(false_negative_rate, 4))
    print()
    print("Classification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["normal/no-fault", "fault"],
        zero_division=0
    ))

    return results_df

In [26]:
fault_replay_results = evaluate_replay_windows(
    source_df=fault_df,
    source_file_name="fault_df",
    loaded_export=loaded_export,
    threshold=0.5,
    sleep_time=0.0,
    print_each_window=False,
)

FINAL REPLAY SUMMARY
Source file: fault_df
Threshold: 0.5
Total 1-second windows: 9500
Correct windows: 1273
Incorrect windows: 8227

Confusion matrix counts:
True negatives : 1272
False positives: 0
False negatives: 8227
True positives : 1

Rates:
Accuracy           : 0.134
Precision          : 1.0
Recall             : 0.0001
False positive rate: 0.0
False negative rate: 0.9999

Classification report:
                 precision    recall  f1-score   support

normal/no-fault       0.13      1.00      0.24      1272
          fault       1.00      0.00      0.00      8228

       accuracy                           0.13      9500
      macro avg       0.57      0.50      0.12      9500
   weighted avg       0.88      0.13      0.03      9500



In [20]:
normal_replay_results = evaluate_replay_windows(
    source_df=normal_df,
    source_file_name="normal_df",
    loaded_export=loaded_export,
    threshold=0.5,
    sleep_time=0.0,
    print_each_window=False,
)

FINAL REPLAY SUMMARY
Source file: normal_df
Threshold: 0.5
Total 1-second windows: 16400
Correct windows: 16399
Incorrect windows: 1

Confusion matrix counts:
True negatives : 16399
False positives: 0
False negatives: 1
True positives : 0

Rates:
Accuracy           : 0.9999
Precision          : 0.0
Recall             : 0.0
False positive rate: 0.0
False negative rate: 1.0

Classification report:
                 precision    recall  f1-score   support

normal/no-fault       1.00      1.00      1.00     16399
          fault       0.00      0.00      0.00         1

       accuracy                           1.00     16400
      macro avg       0.50      0.50      0.50     16400
   weighted avg       1.00      1.00      1.00     16400



In [21]:
all_replay_results = pd.concat(
    [fault_replay_results, normal_replay_results],
    ignore_index=True
)

tn, fp, fn, tp = confusion_matrix(
    all_replay_results["true_label"],
    all_replay_results["pred_label"],
    labels=[0, 1]
).ravel()

total = len(all_replay_results)
correct = int(all_replay_results["correct"].sum())
incorrect = total - correct

false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

print("=" * 100)
print("COMBINED NORMAL + FAULT REPLAY SUMMARY")
print("=" * 100)
print("Total 1-second windows:", total)
print("Correct windows:", correct)
print("Incorrect windows:", incorrect)
print()
print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)
print()
print("False positive rate:", round(false_positive_rate, 4))
print("False negative rate:", round(false_negative_rate, 4))
print()
print(classification_report(
    all_replay_results["true_label"],
    all_replay_results["pred_label"],
    labels=[0, 1],
    target_names=["normal/no-fault", "fault"],
    zero_division=0
))

COMBINED NORMAL + FAULT REPLAY SUMMARY
Total 1-second windows: 25900
Correct windows: 17672
Incorrect windows: 8228

True negatives : 17671
False positives: 0
False negatives: 8228
True positives : 1

False positive rate: 0.0
False negative rate: 0.9999

                 precision    recall  f1-score   support

normal/no-fault       0.68      1.00      0.81     17671
          fault       1.00      0.00      0.00      8229

       accuracy                           0.68     25900
      macro avg       0.84      0.50      0.41     25900
   weighted avg       0.78      0.68      0.55     25900



## 13. Tail a live appended file

This is the runtime behavior you described:

> read from the log file every second and classify only the lines appended during the last second.

For real SmartNIC deployment, the C application would do similar logic:

1. keep a file offset
2. read newly appended bytes/lines
3. aggregate for 1 second
4. extract fixed features
5. run logistic regression equation
6. emit alert if probability crosses threshold

In [27]:
def follow_file_last_second(path: str, poll_interval: float = 1.0, start_at_end: bool = True):
    path = Path(path)
    with path.open("r", errors="ignore") as f:
        if start_at_end:
            f.seek(0, os.SEEK_END)

        while True:
            new_lines = []
            start = time.time()

            while time.time() - start < poll_interval:
                line = f.readline()
                if line:
                    new_lines.append(line.rstrip("\n"))
                else:
                    time.sleep(0.05)

            yield new_lines

In [16]:
# Example live loop.
# Stop manually in Jupyter when done.

# LIVE_LOG_PATH = FAULT_LOG_PATH
# THRESHOLD = 0.70
#
# for new_lines in follow_file_last_second(LIVE_LOG_PATH, poll_interval=1.0, start_at_end=True):
#     if not new_lines:
#         continue
#
#     result = classify_lines(new_lines, loaded_export, threshold=THRESHOLD)
#
#     print("=" * 100)
#     print("Lines appended in last second:", len(new_lines))
#     print("Prediction:", result["prediction"])
#     print("Fault probability:", round(result["fault_probability"], 4))
#
#     if result["fault_probability"] >= THRESHOLD:
#         print("ALERT: possible ICS fault detected")
#         print("\n".join(new_lines[:10]))

## 14. Generate a C header skeleton

This cell writes a header file containing model constants. You still need to implement the C feature extraction function that produces the features in the same order.

The core inference equation is:

```c
score = bias;
for each feature i:
    x_scaled = (x[i] - mean[i]) / scale[i];
    score += weight[i] * x_scaled;
prob = 1.0 / (1.0 + exp(-score));
```

In [28]:
def c_array(name, values, c_type="double"):
    body = ", ".join(f"{float(v):.17g}" for v in values)
    return f"static const {c_type} {name}[] = {{{body}}};"

C_HEADER_PATH = OUTPUT_DIR / "syslog_fault_model.h"

header = []
header.append("#ifndef SYSLOG_FAULT_MODEL_H")
header.append("#define SYSLOG_FAULT_MODEL_H")
header.append("")
header.append("#include <math.h>")
header.append("")
header.append(f"#define SYSLOG_MODEL_NUM_FEATURES {len(feature_names)}")
header.append(f"#define SYSLOG_MODEL_THRESHOLD {float(loaded_export['threshold']):.17g}")
header.append(f"#define SYSLOG_MODEL_BIAS {float(loaded_export['bias']):.17g}")
header.append("")
header.append(c_array("SYSLOG_MODEL_MEAN", loaded_export["scaler_mean"]))
header.append(c_array("SYSLOG_MODEL_SCALE", loaded_export["scaler_scale"]))
header.append(c_array("SYSLOG_MODEL_WEIGHT", loaded_export["weights"]))
header.append("")
header.append("static inline double syslog_sigmoid(double x) {")
header.append("    if (x >= 0.0) {")
header.append("        double z = exp(-x);")
header.append("        return 1.0 / (1.0 + z);")
header.append("    } else {")
header.append("        double z = exp(x);")
header.append("        return z / (1.0 + z);")
header.append("    }")
header.append("}")
header.append("")
header.append("static inline double syslog_predict_fault_probability(const double features[SYSLOG_MODEL_NUM_FEATURES]) {")
header.append("    double score = SYSLOG_MODEL_BIAS;")
header.append("    for (int i = 0; i < SYSLOG_MODEL_NUM_FEATURES; i++) {")
header.append("        double x_scaled = 0.0;")
header.append("        if (SYSLOG_MODEL_SCALE[i] != 0.0) {")
header.append("            x_scaled = (features[i] - SYSLOG_MODEL_MEAN[i]) / SYSLOG_MODEL_SCALE[i];")
header.append("        }")
header.append("        score += SYSLOG_MODEL_WEIGHT[i] * x_scaled;")
header.append("    }")
header.append("    return syslog_sigmoid(score);")
header.append("}")
header.append("")
header.append("static inline int syslog_predict_fault_label(const double features[SYSLOG_MODEL_NUM_FEATURES]) {")
header.append("    return syslog_predict_fault_probability(features) >= SYSLOG_MODEL_THRESHOLD ? 1 : 0;")
header.append("}")
header.append("")
header.append("#endif")
header.append("")

C_HEADER_PATH.write_text("\n".join(header))
print("Wrote:", C_HEADER_PATH)

Wrote: syslog_fault_outputs/syslog_fault_model.h


## 15. C implementation notes for SmartNIC CPU cores

Recommended deployment architecture:

```text
log source / ring buffer / file tail
        ↓
1-second line aggregator
        ↓
C feature extractor
        ↓
standardized logistic regression
        ↓
normal/fault decision + probability
        ↓
alert, telemetry, or packet/log forwarding decision
```

Porting checklist:

1. Keep the exact feature order from `feature_order.txt`.
2. Reimplement `extract_c_portable_features()` in C.
3. Include `syslog_fault_model.h`.
4. Fill a `double features[SYSLOG_MODEL_NUM_FEATURES]` array once per second.
5. Call `syslog_predict_fault_probability(features)`.
6. Alert when probability is above your chosen threshold, for example `0.70`.

Why this is SmartNIC-friendly:

- no Python runtime required
- no transformer runtime required
- no dynamic memory requirement if implemented carefully
- no tokenization model
- bounded CPU cost per line/window
- easy to audit and debug

A transformer/small-language-model can still be useful offline for comparison, but the exported logistic regression path is the practical path for C on SmartNIC CPU cores.

## 16. Optional: TF-IDF text baseline

This may perform well, but it is less convenient to port to C than fixed numeric features because you need to also port tokenization, vocabulary lookup, and sparse-vector math.

In [29]:
text_X = window_df["window_text"].astype(str)
text_y = window_df["label"].astype(int).values

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    text_X,
    text_y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=text_y if len(np.unique(text_y)) > 1 else None,
)

text_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

text_model.fit(X_text_train, y_text_train)
text_pred = text_model.predict(X_text_test)

print(classification_report(y_text_test, text_pred, target_names=["normal", "fault"]))
print(confusion_matrix(y_text_test, text_pred))

              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      4100
       fault       1.00      1.00      1.00      2375

    accuracy                           1.00      6475
   macro avg       1.00      1.00      1.00      6475
weighted avg       1.00      1.00      1.00      6475

[[4099    1]
 [   0 2375]]


## 17. Optional: small language model research path

This is included for experimentation only. I do **not** recommend this as the first SmartNIC C deployment target.

Reasons:

- much higher CPU and memory cost
- tokenizer is non-trivial to port
- model runtime is non-trivial to port
- harder real-time guarantees

Use this only to compare whether a transformer meaningfully improves detection over the C-portable model.

In [30]:
# Optional transformer experiment.
# Leave this cell commented unless you are using a clean environment/kernel created specifically for Hugging Face.
# This section is for research comparison only. It is NOT the deployment path for the SmartNIC C application.

# If your environment shows errors such as:
#   huggingface-hub requires packaging>=20.9
#   datasets requires requests>=2.32.2
# then do not try to force-install packages into the old system environment.
# Use the venv instructions in Section 1, or skip this cell.

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

MODEL_NAME = "distilbert-base-uncased"
hf_df = window_df[["window_text", "label"]].rename(columns={"window_text": "text"})
hf_df["label"] = hf_df["label"].astype(int)

train_df, test_df = train_test_split(
    hf_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=hf_df["label"] if hf_df["label"].nunique() > 1 else None,
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])
train_ds.set_format("torch")
test_ds.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./slm_syslog_fault_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()


/home/ubuntu/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 6475/6475 [00:02<00:00, 2522.40 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_47673/978259115.py:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended t

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.036400,0.011795,0.995212,0.999574,0.987368,0.993434
2,0.000000,0.001293,0.999846,0.999579,1.000000,0.999790
3,0.000000,0.001380,0.999846,0.999579,1.000000,0.999790


{'eval_loss': 0.0012925963383167982,
 'eval_accuracy': 0.9998455598455599,
 'eval_precision': 0.9995791245791246,
 'eval_recall': 1.0,
 'eval_f1': 0.9997895179962113,
 'eval_runtime': 217.9003,
 'eval_samples_per_second': 29.715,
 'eval_steps_per_second': 3.717,
 'epoch': 3.0}

In [31]:
SLM_SAVE_DIR = "./slm_syslog_fault_model"

trainer.save_model(SLM_SAVE_DIR)
tokenizer.save_pretrained(SLM_SAVE_DIR)

print("Saved SLM model to:", SLM_SAVE_DIR)

Saved SLM model to: ./slm_syslog_fault_model


In [32]:
def classify_lines_slm(lines, threshold=0.5):
    text = "\n".join(lines)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    fault_probability = float(probs[1])
    prediction = "fault" if fault_probability >= threshold else "normal"

    return {
        "prediction": prediction,
        "normal_probability": float(probs[0]),
        "fault_probability": fault_probability,
    }

In [33]:
for second, new_lines in replay_log_by_second(fault_df, sleep_time=0.0):
    result = classify_lines_slm(new_lines)

    print("Window:", second)
    print("Prediction:", result["prediction"])
    print("Fault probability:", round(result["fault_probability"], 4))
    print("Sample lines:")
    print("\n".join(new_lines[:3]))
    break

Window: 2025-01-09 19:22:32+00:00
Prediction: fault
Fault probability: 0.9997
Sample lines:
2025-01-10T03:22:32+08:00 smasher-virtualbox attackerremote[826]: [#033attacker_remote#033 - #03303:22:32#033[0m]#011#033[INFO] Created#033#015
2025-01-10T03:22:32+08:00 smasher-virtualbox attackerremote[826]: AttackerRemote.py:25: DeprecationWarning: Callback API version 1 is deprecated, update to latest version#015
2025-01-10T03:22:32+08:00 smasher-virtualbox attackerremote[826]:   self.client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION1)#015


## Testing

In [35]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, classification_report

# Make sure the trained Hugging Face model is on the right device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

FAULT_EVENT_KEYWORDS = [
    "fault applied",
    "sensor drift applied",
    "tank leak fault",
    "conveyor belt sticking fault",
    "memory corruption fault",
    "actuator failure",
    "fault",
    "critical",
    "error",
]

def infer_window_ground_truth(lines):
    """
    Keyword-inferred window label:
      1 = fault evidence appears in this 1-second window
      0 = no explicit fault evidence in this 1-second window
    """
    text = "\n".join(lines).lower()
    return 1 if any(k in text for k in FAULT_EVENT_KEYWORDS) else 0


def classify_lines_slm_timed(lines, threshold=0.5, max_length=256):
    """
    Classify one 1-second log window using the trained DistilBERT SLM.

    Returns:
      prediction
      normal_probability
      fault_probability
      inference_time_ms
    """

    text = "\n".join(lines)

    # Synchronize before timing when using GPU.
    # This makes timing more accurate because CUDA operations are asynchronous.
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    normal_probability = float(probs[0].detach().cpu())
    fault_probability = float(probs[1].detach().cpu())

    prediction = "fault" if fault_probability >= threshold else "normal"

    return {
        "prediction": prediction,
        "pred_label": 1 if prediction == "fault" else 0,
        "normal_probability": normal_probability,
        "fault_probability": fault_probability,
        "inference_time_ms": (end - start) * 1000.0,
    }


def evaluate_slm_replay_windows(
    source_df,
    source_file_name,
    threshold=0.5,
    sleep_time=0.0,
    max_length=256,
    print_each_window=False,
    max_sample_lines=3,
):
    """
    Replay all 1-second windows from a parsed log dataframe and evaluate the SLM.

    Ground truth:
      Uses infer_window_ground_truth(lines), not the file-level label.
      This is better because the beginning of the fault file may contain normal startup logs.

    Returns:
      results_df with one row per 1-second window.
    """

    rows = []

    for second, new_lines in replay_log_by_second(source_df, sleep_time=sleep_time):
        if not new_lines:
            continue

        result = classify_lines_slm_timed(
            lines=new_lines,
            threshold=threshold,
            max_length=max_length,
        )

        true_label = infer_window_ground_truth(new_lines)
        pred_label = result["pred_label"]

        if true_label == 1 and pred_label == 1:
            outcome = "true_positive"
        elif true_label == 0 and pred_label == 0:
            outcome = "true_negative"
        elif true_label == 0 and pred_label == 1:
            outcome = "false_positive"
        else:
            outcome = "false_negative"

        row = {
            "second": second,
            "source_file": source_file_name,
            "num_lines": len(new_lines),
            "true_label": true_label,
            "true_name": "fault" if true_label == 1 else "normal/no-fault",
            "pred_label": pred_label,
            "prediction": result["prediction"],
            "normal_probability": result["normal_probability"],
            "fault_probability": result["fault_probability"],
            "inference_time_ms": result["inference_time_ms"],
            "correct": pred_label == true_label,
            "outcome": outcome,
            "sample_lines": "\n".join(new_lines[:max_sample_lines]),
        }

        rows.append(row)

        if print_each_window:
            print("=" * 100)
            print("Window:", second)
            print("Source file:", source_file_name)
            print("Window ground truth:", row["true_name"])
            print("Lines:", len(new_lines))
            print("Prediction:", result["prediction"])
            print("Correct:", row["correct"])
            print("Fault probability:", round(result["fault_probability"], 4))
            print("Inference time ms:", round(result["inference_time_ms"], 3))
            print("Outcome:", outcome)
            print("Sample lines:")
            print(row["sample_lines"])

    return pd.DataFrame(rows)


def print_slm_summary(results_df, title="SLM REPLAY SUMMARY"):
    """
    Print evaluation and timing summary for replay results.
    """

    if results_df.empty:
        print("No windows were evaluated.")
        return

    y_true = results_df["true_label"]
    y_pred = results_df["pred_label"]

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    total = len(results_df)
    correct = int(results_df["correct"].sum())
    incorrect = total - correct

    accuracy = correct / total if total > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    avg_time = results_df["inference_time_ms"].mean()
    median_time = results_df["inference_time_ms"].median()
    p95_time = results_df["inference_time_ms"].quantile(0.95)
    min_time = results_df["inference_time_ms"].min()
    max_time = results_df["inference_time_ms"].max()

    windows_per_second = 1000.0 / avg_time if avg_time > 0 else float("inf")

    print("=" * 100)
    print(title)
    print("=" * 100)
    print("Total 1-second windows:", total)
    print("Correct windows:", correct)
    print("Incorrect windows:", incorrect)
    print()
    print("Confusion matrix counts:")
    print("True negatives :", tn)
    print("False positives:", fp)
    print("False negatives:", fn)
    print("True positives :", tp)
    print()
    print("Rates:")
    print("Accuracy            :", round(accuracy, 6))
    print("Precision           :", round(precision, 6))
    print("Recall              :", round(recall, 6))
    print("False positive rate :", round(false_positive_rate, 6))
    print("False negative rate :", round(false_negative_rate, 6))
    print()
    print("Inference timing:")
    print("Average inference time per 1-second window, ms:", round(avg_time, 3))
    print("Median inference time, ms:", round(median_time, 3))
    print("95th percentile inference time, ms:", round(p95_time, 3))
    print("Minimum inference time, ms:", round(min_time, 3))
    print("Maximum inference time, ms:", round(max_time, 3))
    print("Approx windows processed per second:", round(windows_per_second, 2))
    print()
    print("Classification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["normal/no-fault", "fault"],
        zero_division=0,
    ))

In [36]:
fault_slm_results = evaluate_slm_replay_windows(
    source_df=fault_df,
    source_file_name="fault_df",
    threshold=0.5,
    sleep_time=0.0,
    max_length=256,
    print_each_window=False,
)

print_slm_summary(
    fault_slm_results,
    title="SLM REPLAY SUMMARY: FAULT LOG"
)

SLM REPLAY SUMMARY: FAULT LOG
Total 1-second windows: 9500
Correct windows: 8228
Incorrect windows: 1272

Confusion matrix counts:
True negatives : 0
False positives: 1272
False negatives: 0
True positives : 8228

Rates:
Accuracy            : 0.866105
Precision           : 0.866105
Recall              : 1.0
False positive rate : 1.0
False negative rate : 0.0

Inference timing:
Average inference time per 1-second window, ms: 71.8
Median inference time, ms: 70.652
95th percentile inference time, ms: 77.587
Minimum inference time, ms: 58.932
Maximum inference time, ms: 812.47
Approx windows processed per second: 13.93

Classification report:
                 precision    recall  f1-score   support

normal/no-fault       0.00      0.00      0.00      1272
          fault       0.87      1.00      0.93      8228

       accuracy                           0.87      9500
      macro avg       0.43      0.50      0.46      9500
   weighted avg       0.75      0.87      0.80      9500



In [37]:
normal_slm_results = evaluate_slm_replay_windows(
    source_df=normal_df,
    source_file_name="normal_df",
    threshold=0.5,
    sleep_time=0.0,
    max_length=256,
    print_each_window=False,
)

print_slm_summary(
    normal_slm_results,
    title="SLM REPLAY SUMMARY: NORMAL LOG"
)

SLM REPLAY SUMMARY: NORMAL LOG
Total 1-second windows: 16400
Correct windows: 2
Incorrect windows: 16398

Confusion matrix counts:
True negatives : 1
False positives: 16398
False negatives: 0
True positives : 1

Rates:
Accuracy            : 0.000122
Precision           : 6.1e-05
Recall              : 1.0
False positive rate : 0.999939
False negative rate : 0.0

Inference timing:
Average inference time per 1-second window, ms: 75.465
Median inference time, ms: 75.091
95th percentile inference time, ms: 82.93
Minimum inference time, ms: 31.135
Maximum inference time, ms: 791.228
Approx windows processed per second: 13.25

Classification report:
                 precision    recall  f1-score   support

normal/no-fault       1.00      0.00      0.00     16399
          fault       0.00      1.00      0.00         1

       accuracy                           0.00     16400
      macro avg       0.50      0.50      0.00     16400
   weighted avg       1.00      0.00      0.00     16400



In [38]:
all_slm_results = pd.concat(
    [fault_slm_results, normal_slm_results],
    ignore_index=True,
)

print_slm_summary(
    all_slm_results,
    title="SLM REPLAY SUMMARY: FAULT + NORMAL LOGS"
)

SLM REPLAY SUMMARY: FAULT + NORMAL LOGS
Total 1-second windows: 25900
Correct windows: 8230
Incorrect windows: 17670

Confusion matrix counts:
True negatives : 1
False positives: 17670
False negatives: 0
True positives : 8229

Rates:
Accuracy            : 0.317761
Precision           : 0.317734
Recall              : 1.0
False positive rate : 0.999943
False negative rate : 0.0

Inference timing:
Average inference time per 1-second window, ms: 74.121
Median inference time, ms: 73.467
95th percentile inference time, ms: 82.158
Minimum inference time, ms: 31.135
Maximum inference time, ms: 812.47
Approx windows processed per second: 13.49

Classification report:
                 precision    recall  f1-score   support

normal/no-fault       1.00      0.00      0.00     17671
          fault       0.32      1.00      0.48      8229

       accuracy                           0.32     25900
      macro avg       0.66      0.50      0.24     25900
   weighted avg       0.78      0.32      0.15

In [39]:
false_positives_slm = all_slm_results[
    all_slm_results["outcome"] == "false_positive"
]

false_negatives_slm = all_slm_results[
    all_slm_results["outcome"] == "false_negative"
]

print("False positives:", len(false_positives_slm))
display(false_positives_slm[[
    "second",
    "source_file",
    "num_lines",
    "fault_probability",
    "inference_time_ms",
    "sample_lines",
]].head(20))

print("False negatives:", len(false_negatives_slm))
display(false_negatives_slm[[
    "second",
    "source_file",
    "num_lines",
    "fault_probability",
    "inference_time_ms",
    "sample_lines",
]].head(20))

False positives: 17670


,second,source_file,num_lines,fault_probability,inference_time_ms,sample_lines
0,2025-01-09 19:22:32+00:00,fault_df,20,0.999705,545.124839,2025-01-10T03:22:32+08:00 smasher-virtualbox a...
1,2025-01-09 19:22:33+00:00,fault_df,26,0.999745,77.831297,2025-01-10T03:22:33+08:00 smasher-virtualbox h...
4,2025-01-09 19:22:36+00:00,fault_df,39,0.991389,70.020187,2025-01-10T03:22:36+08:00 smasher-virtualbox h...
6,2025-01-09 19:22:38+00:00,fault_df,38,0.964486,69.492548,2025-01-10T03:22:38+08:00 smasher-virtualbox h...
9,2025-01-09 19:22:41+00:00,fault_df,38,0.992670,68.982942,2025-01-10T03:22:41+08:00 smasher-virtualbox h...
14,2025-01-09 19:22:46+00:00,fault_df,38,0.999333,70.153640,2025-01-10T03:22:46+08:00 smasher-virtualbox h...
19,2025-01-09 19:22:51+00:00,fault_df,38,0.991909,74.674427,2025-01-10T03:22:51+08:00 smasher-virtualbox h...
22,2025-01-09 19:22:54+00:00,fault_df,38,0.991269,73.004032,2025-01-10T03:22:54+08:00 smasher-virtualbox h...
27,2025-01-09 19:22:59+00:00,fault_df,38,0.965608,64.511651,2025-01-10T03:22:59+08:00 smasher-virtualbox h...
30,2025-01-09 19:23:02+00:00,fault_df,43,0.993318,76.612600,2025-01-10T03:23:02+08:00 smasher-virtualbox h...


False negatives: 0


,second,source_file,num_lines,fault_probability,inference_time_ms,sample_lines
